<a href="https://colab.research.google.com/github/melissa-04/melisayla-biyoinformatik/blob/main/notebooks/scrna/01_veri_mutfagi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Veri mutfağı: PBMC3k

İkinci proje burada başlıyor. Veri, alanın klasiği: 10x Genomics'in halka açık PBMC3k seti — sağlıklı bir bağışçının periferik kanından yaklaşık 2.700 mononükleer hücre. Birinci projede altı örneğin ortalamasını okuduk; burada her hücre kendi satırı olacak. Bu defterin işi analiz değil mutfak: veriyi kaynağından al, tanı, kaba temizliğini yap ve arşivlik dosyayı üret.

In [1]:
%pip install -q scanpy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 46.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.8/188.8 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.0/40.0 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 96.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.1/79.1 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.7/363.7 kB 26.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 69.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 5.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.0.5 which is incompatible.


## 1. Kaynak

Birincil kaynak 10x Genomics'in halka açık veri sayfasıdır (CC-BY lisansı). Ders kolaylığı için üç dosyayı bir topluluk aynasından indiriyoruz: barkod listesi, gen listesi ve seyrek matris. Bu rehberin sonunda ürettiğimiz dosyayı Zenodo'ya yükleyeceğiz; 2.2'den itibaren defterler veriyi oradan, DOI'li kalıcı adresten okuyacak.

In [2]:
import warnings; warnings.filterwarnings('ignore')
import urllib.request, gzip, shutil, os

os.makedirs('pbmc/hg19', exist_ok=True)
kok = 'https://raw.githubusercontent.com/Amartya101/PBMC3k_Data/main/PBMC3k/'
esle = {'10X_PBMC3k_barcodes.tsv': 'barcodes.tsv',
        '10X_PBMC3k_features.tsv': 'genes.tsv',
        '10X_PBMC3k_matrix.mtx.gz': 'matrix.mtx.gz'}
for kaynak, hedef in esle.items():
    urllib.request.urlretrieve(kok + kaynak, 'pbmc/hg19/' + hedef)
with gzip.open('pbmc/hg19/matrix.mtx.gz', 'rb') as f_in, open('pbmc/hg19/matrix.mtx', 'wb') as f_out:
    shutil.copyfileobj(f_in, f_out)
os.remove('pbmc/hg19/matrix.mtx.gz')

import scanpy as sc
a = sc.read_10x_mtx('pbmc/hg19', var_names='gene_symbols', make_unique=True)
print('hücre × gen:', a.shape)

hücre × gen: (2700, 32738)


## 2. Matrisi tanımak

Satırlar artık örnek değil, tek tek hücreler: her satırın adı, o hücreyi damlacığında etiketleyen DNA barkodu. Sütunlar gen; gen listesi dosyasında hem kimlik hem ad var ve adlar benzersiz olmadığı için `make_unique` kullandık — birinci serinin altıncı rehberindeki ders burada da geçerli. Aşağıda hücre başına üç temel ölçüye bakıyoruz: toplam UMI (hücrenin toplam sesi), tespit edilen gen sayısı ve mitokondriyal okuma yüzdesi.

In [3]:
import numpy as np
a.var['mt'] = a.var_names.str.startswith('MT-')
print('Mitokondriyal gen:', int(a.var['mt'].sum()))

sc.pp.calculate_qc_metrics(a, qc_vars=['mt'], inplace=True, percent_top=None, log1p=False)
print('Medyan UMI:', int(np.median(a.obs.total_counts)),
      '| medyan gen:', int(np.median(a.obs.n_genes_by_counts)))
print('Medyan MT%%: %.2f' % a.obs.pct_counts_mt.median(),
      '| MT%>10 hücre:', int((a.obs.pct_counts_mt > 10).sum()),
      '| gen>2500 hücre:', int((a.obs.n_genes_by_counts > 2500).sum()))

Mitokondriyal gen: 13
Medyan UMI: 2197 | medyan gen: 817
Medyan MT%: 2.03 | MT%>10 hücre: 6 | gen>2500 hücre: 5


## 3. Üç filtre, üç gerekçe

En az 200 gen: neredeyse boş damlacıklar hücre değil gürültüdür. Gen başına en az 3 hücre: kimsenin ifade etmediği hayalet sütunlar tabloyu şişirir. MT% sınırı: zarı hasarlanan hücrenin sitoplazmik RNA'sı dışarı kaçar ama mitokondri içeride kalır; yüksek MT%, ölmekte olan hücrenin imzasıdır. Eşik dokuya göre değişir; PBMC için %10 yaygın bir tercihtir.

In [4]:
once = a.shape
sc.pp.filter_cells(a, min_genes=200)
sc.pp.filter_genes(a, min_cells=3)
a = a[a.obs.pct_counts_mt < 10].copy()
print('önce:', once, '-> sonra:', a.shape)

a.write('pbmc3k_ders.h5ad')
print('pbmc3k_ders.h5ad: %.1f MB' % (os.path.getsize('pbmc3k_ders.h5ad') / 1e6))

önce: (2700, 32738) -> sonra: (2694, 13714)
pbmc3k_ders.h5ad: 20.2 MB


## Kendin dene

Üç görev. Birincisi: kendi koşunuzdan filtre öncesi ve sonrası boyutları, medyan UMI ve medyan gen sayısını not edin. İkincisi: MT%>10 ve gen>2500 ölçütlerine takılan hücre sayılarını yazın ve tek cümleyle söyleyin: yüksek MT% neden "ölmekte olan hücre" demektir? Üçüncüsü bu serinin ilk mührü ve bir sayı değil: zenodo.org'a GitHub hesabınızla girin, `pbmc3k_ders.h5ad` dosyasını yeni bir kayıt olarak yükleyin — başlıkta proje adı, açıklamada 10x Genomics atfı ve CC-BY lisansı — Publish deyin ve aldığınız **DOI'yi** yazın. 2.2'den itibaren bütün defterler veriyi o DOI'den okuyacak.